# SARIMAX Experiments


## Setup


In [1]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repo root (pyproject.toml). Open this notebook from the repo.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT


WindowsPath('C:/Users/baben_bakg1j1/HSE/annual_project/stocks-advisor')

In [2]:
import json
import pickle
import tempfile
from copy import deepcopy
from pathlib import Path

import mlflow
import mlflow.statsmodels
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

from jupyter_utils import setup_jupyter_notebook
from stocks_dl.constants import TARGET_COLUMN
from stocks_dl.data.pipeline import load_features_multi
from stocks_dl.training.dataset import split_train_test
from stocks_dl.training.train import calculate_metrics

import warnings
warnings.filterwarnings('ignore')


In [3]:
EXPERIMENT_NAME = 'sarimax_checkpoint'
setup_jupyter_notebook(environment='prod', experiment=EXPERIMENT_NAME)

# Для локального запуска:
# setup_jupyter_notebook(environment='local', experiment='sarimax_test')


2026/06/09 13:58:28 INFO mlflow.tracking.fluent: Experiment with name 'sarimax_checkpoint' does not exist. Creating a new experiment.


Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: sarimax_checkpoint
Database: localhost:15432/stocks_advisor_db


In [4]:
TICKERS = ['SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN']
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
ENRICHMENTS_LIMIT = 1_000_000

RUNS_DIR = REPO_ROOT / 'stocks_dl_runs' / 'sarimax'
RUNS_DIR.mkdir(parents=True, exist_ok=True)


## Data


In [5]:
features_by_ticker, enrichments_df = load_features_multi(
    TICKERS,
    enrichments_limit=ENRICHMENTS_LIMIT,
)

for ticker, df in features_by_ticker.items():
    print(ticker, df.shape, df['begin'].min(), df['begin'].max())


SBER (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
TCSG (2853, 89) 2022-07-04 10:00:00 2024-11-20 18:00:00
GAZP (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
LKOH (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
ROSN (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00


## Helpers


In [6]:
def build_sarimax_configs(ticker: str) -> list[dict]:
    base = {
        'seasonal_order': (0, 0, 0, 0),
        'trend': 'c',
        'maxiter': 100,
        'use_exog': False,
        'max_exog_features': 0,
    }
    variants = [
        ('sarima_100', {'order': (1, 0, 0)}),
        ('sarima_200', {'order': (2, 0, 0)}),
        ('sarima_101', {'order': (1, 0, 1)}),
        ('sarimax_100_exog12', {'order': (1, 0, 0), 'use_exog': True, 'max_exog_features': 12}),
        ('sarimax_101_exog12', {'order': (1, 0, 1), 'use_exog': True, 'max_exog_features': 12}),
    ]
    return [
        {
            'run_name': f'{ticker.lower()}_sarimax_{idx:02d}_{name}',
            'ticker': ticker,
            'variant': name,
            **base,
            **params,
        }
        for idx, (name, params) in enumerate(variants, start=1)
    ]


def select_exog_columns(train_df: pd.DataFrame, max_features: int) -> list[str]:
    if max_features <= 0:
        return []

    candidate_cols = [c for c in train_df.columns if c not in ('begin', TARGET_COLUMN)]
    numeric = train_df[candidate_cols].replace([np.inf, -np.inf], np.nan)
    corr = numeric.corrwith(train_df[TARGET_COLUMN]).abs().replace([np.inf, -np.inf], np.nan).dropna()
    ranked = corr.sort_values(ascending=False).index.tolist()

    sentiment_cols = [c for c in candidate_cols if 'sentiment' in c.lower()]
    selected = []
    for col in sentiment_cols + ranked:
        if col not in selected:
            selected.append(col)
        if len(selected) >= max_features:
            break
    return selected


def make_exog_transformer():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])


def prepare_exog(train_df, val_df, test_df, exog_cols):
    if not exog_cols:
        return None, None, None, None
    transformer = make_exog_transformer()
    X_train = train_df[exog_cols].replace([np.inf, -np.inf], np.nan)
    X_val = val_df[exog_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[exog_cols].replace([np.inf, -np.inf], np.nan)

    exog_train = pd.DataFrame(transformer.fit_transform(X_train), columns=exog_cols, index=train_df.index)
    exog_val = pd.DataFrame(transformer.transform(X_val), columns=exog_cols, index=val_df.index)
    exog_test = pd.DataFrame(transformer.transform(X_test), columns=exog_cols, index=test_df.index)
    return exog_train, exog_val, exog_test, transformer


def predictions_frame(test_df: pd.DataFrame, y_true, y_pred) -> pd.DataFrame:
    out = pd.DataFrame({
        'begin': pd.to_datetime(test_df['begin']).reset_index(drop=True),
        TARGET_COLUMN: np.asarray(y_true),
        'predict': np.asarray(y_pred),
    })
    out['error'] = out['predict'] - out[TARGET_COLUMN]
    out['abs_error'] = out['error'].abs()
    out['direction_match'] = np.sign(out['predict']) == np.sign(out[TARGET_COLUMN])
    return out


def safe_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'direction_accuracy': np.nan}
    return calculate_metrics(y_true[mask], y_pred[mask])


In [7]:
def log_sarimax_model(result, artifact_path: str = 'model') -> None:
    try:
        mlflow.statsmodels.log_model(result, artifact_path)
    except Exception as exc:
        print(f'mlflow.statsmodels.log_model failed: {exc}. Saving pickle artifact instead.')
        with tempfile.TemporaryDirectory() as tmpdir:
            path = Path(tmpdir) / 'sarimax_result.pkl'
            with path.open('wb') as f:
                pickle.dump(result, f)
            mlflow.log_artifact(str(path), artifact_path=artifact_path)


def run_sarimax_experiment(
    cfg: dict,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    params = deepcopy(cfg)
    run_name = params['run_name']
    ticker = params['ticker']
    variant = params['variant']
    order = tuple(params['order'])
    seasonal_order = tuple(params['seasonal_order'])
    trend = params['trend']
    maxiter = int(params['maxiter'])
    exog_cols = select_exog_columns(train_df, int(params['max_exog_features'])) if params['use_exog'] else []
    exog_train, exog_val, exog_test, transformer = prepare_exog(train_df, val_df, test_df, exog_cols)

    y_train = train_df[TARGET_COLUMN].astype(float).reset_index(drop=True)
    y_val = val_df[TARGET_COLUMN].astype(float).reset_index(drop=True)
    y_test = test_df[TARGET_COLUMN].astype(float).reset_index(drop=True)

    model = SARIMAX(
        y_train,
        exog=exog_train.reset_index(drop=True) if exog_train is not None else None,
        order=order,
        seasonal_order=seasonal_order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    with tempfile.TemporaryDirectory() as tmpdir, mlflow.start_run(run_name=run_name) as run:
        tmpdir = Path(tmpdir)
        mlflow.set_tags({
            'ticker': ticker,
            'stage': 'sarimax_search',
            'model_family': 'SARIMAX',
            'target': TARGET_COLUMN,
            'variant': variant,
            'seed': str(SEED),
        })
        mlflow.log_params({
            'variant': variant,
            'order': str(order),
            'seasonal_order': str(seasonal_order),
            'trend': trend,
            'maxiter': maxiter,
            'use_exog': bool(exog_cols),
            'exog_count': len(exog_cols),
        })
        mlflow.log_dict({**cfg, 'order': list(order), 'seasonal_order': list(seasonal_order), 'exog_cols': exog_cols}, 'config.json')

        result = model.fit(disp=False, maxiter=maxiter)

        train_pred = np.asarray(result.fittedvalues, dtype=float)
        forecast_exog = None
        if exog_cols:
            forecast_exog = pd.concat([exog_val, exog_test], axis=0).reset_index(drop=True)
        forecast = result.get_forecast(steps=len(val_df) + len(test_df), exog=forecast_exog)
        forecast_values = np.asarray(forecast.predicted_mean, dtype=float)
        val_pred = forecast_values[: len(val_df)]
        test_pred = forecast_values[len(val_df) :]

        train_metrics = safe_metrics(y_train.to_numpy(), train_pred)
        val_metrics = safe_metrics(y_val.to_numpy(), val_pred)
        test_metrics = safe_metrics(y_test.to_numpy(), test_pred)

        pred_df = predictions_frame(test_df, y_test.to_numpy(), test_pred)
        pred_df.to_csv(tmpdir / 'test_predictions.csv', index=False)

        pd.Series(result.params).to_csv(tmpdir / 'model_params.csv')
        (tmpdir / 'model_summary.txt').write_text(str(result.summary()), encoding='utf-8')
        (tmpdir / 'exog_cols.json').write_text(json.dumps(exog_cols, ensure_ascii=False, indent=2), encoding='utf-8')

        mlflow.log_artifacts(str(tmpdir))
        log_sarimax_model(result)

        summary = {
            'run_id': run.info.run_id,
            'run_name': run_name,
            'ticker': ticker,
            'variant': variant,
            'order': str(order),
            'seasonal_order': str(seasonal_order),
            'trend': trend,
            'use_exog': bool(exog_cols),
            'exog_count': len(exog_cols),
            'aic': float(result.aic),
            'bic': float(result.bic),
            'train_mae': float(train_metrics['mae']),
            'train_rmse': float(train_metrics['rmse']),
            'train_r2': float(train_metrics['r2']),
            'train_direction_accuracy': float(train_metrics['direction_accuracy']),
            'val_mae': float(val_metrics['mae']),
            'val_rmse': float(val_metrics['rmse']),
            'val_r2': float(val_metrics['r2']),
            'val_direction_accuracy': float(val_metrics['direction_accuracy']),
            'test_mae': float(test_metrics['mae']),
            'test_rmse': float(test_metrics['rmse']),
            'test_r2': float(test_metrics['r2']),
            'test_direction_accuracy': float(test_metrics['direction_accuracy']),
            'config_json': json.dumps({**cfg, 'order': list(order), 'seasonal_order': list(seasonal_order), 'exog_cols': exog_cols}, ensure_ascii=False),
        }
        mlflow.log_metrics({k: v for k, v in summary.items() if isinstance(v, (int, float, np.floating)) and np.isfinite(v)})

    print(f'[{run_name}] val_dir_acc={summary["val_direction_accuracy"]:.4f} test_dir_acc={summary["test_direction_accuracy"]:.4f}')
    return summary


## Experiments


In [8]:
mlflow.set_experiment(EXPERIMENT_NAME)

all_rows = []
ticker_summaries = {}

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')
    features_df = features_by_ticker[ticker].copy()
    train_df, val_df, test_df = split_train_test(
        features_df,
        target_column=TARGET_COLUMN,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
    )

    rows = []
    for cfg in build_sarimax_configs(ticker):
        rows.append(run_sarimax_experiment(cfg, train_df, val_df, test_df))

    summary_df = (
        pd.DataFrame(rows)
        .sort_values(['val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    ticker_summaries[ticker] = summary_df
    all_rows.extend(rows)

    out_path = RUNS_DIR / f'{ticker.lower()}_sarimax_summary.csv'
    summary_df.to_csv(out_path, index=False)
    display(summary_df)



=== SBER ===


2026/06/09 14:01:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 14:01:43 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:01:43 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:01:43 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 14:01:43 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 14:01:43 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 14:01:45 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run sber_sarimax_01_sarima_100 at: http://localhost:5050/#/experiments/7/runs/725575de9e314be09be8935f3c691656
🧪 View experiment at: http://localhost:5050/#/experiments/7
[sber_sarimax_01_sarima_100] val_dir_acc=0.4702 test_dir_acc=0.5324


2026/06/09 14:01:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 14:01:47 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:01:47 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:01:47 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 14:01:47 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 14:01:47 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 14:01:48 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run sber_sarimax_02_sarima_200 at: http://localhost:5050/#/experiments/7/runs/0c2d63c1f4964c4982b7146fb50431b7
🧪 View experiment at: http://localhost:5050/#/experiments/7
[sber_sarimax_02_sarima_200] val_dir_acc=0.4726 test_dir_acc=0.5324


...

2026/06/09 14:04:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 14:04:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:04:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:04:15 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 14:04:15 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 14:04:15 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 14:04:16 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run rosn_sarimax_04_sarimax_100_exog12 at: http://localhost:5050/#/experiments/7/runs/e0f94d9ce8f241e99899e22a2485abc9
🧪 View experiment at: http://localhost:5050/#/experiments/7
[rosn_sarimax_04_sarimax_100_exog12] val_dir_acc=0.6119 test_dir_acc=0.4552


2026/06/09 14:04:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 14:04:28 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:04:28 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 14:04:28 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 14:04:28 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 14:04:28 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 14:04:29 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run rosn_sarimax_05_sarimax_101_exog12 at: http://localhost:5050/#/experiments/7/runs/afa65977aa5d497b99c9eee59f073c11
🧪 View experiment at: http://localhost:5050/#/experiments/7
[rosn_sarimax_05_sarimax_101_exog12] val_dir_acc=0.6179 test_dir_acc=0.4514


,run_id,run_name,ticker,variant,order,seasonal_order,trend,use_exog,exog_count,aic,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,afa65977aa5d497b99c9eee59f073c11,rosn_sarimax_05_sarimax_101_exog12,ROSN,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,10105.105827,...,0.919595,4.248948,5.876684,-0.717475,0.617857,5.130426,6.616807,-1.689025,0.451429,"{""run_name"": ""rosn_sarimax_05_sarimax_101_exog..."
1,e0f94d9ce8f241e99899e22a2485abc9,rosn_sarimax_04_sarimax_100_exog12,ROSN,sarimax_100_exog12,"(1, 0, 0)","(0, 0, 0, 0)",c,True,12,10102.791210,...,0.918999,4.376609,5.965552,-0.769812,0.611905,5.047209,6.518316,-1.609569,0.455238,"{""run_name"": ""rosn_sarimax_04_sarimax_100_exog..."
2,8a2e189301904548a4055f36fe2b928e,rosn_sarimax_01_sarima_100,ROSN,sarima_100,"(1, 0, 0)","(0, 0, 0, 0)",c,False,0,10223.961444,...,0.923466,3.635702,4.546195,-0.027831,0.488095,3.013709,4.104229,-0.034573,0.413333,"{""run_name"": ""rosn_sarimax_01_sarima_100"", ""ti..."
3,87d37b4d8d664a98a6e7a772bb053883,rosn_sarimax_03_sarima_101,ROSN,sarima_101,"(1, 0, 1)","(0, 0, 0, 0)",c,False,0,10222.929907,...,0.922871,3.633637,4.543971,-0.026826,0.485714,3.013871,4.104346,-0.034632,0.413333,"{""run_name"": ""rosn_sarimax_03_sarima_101"", ""ti..."
4,ffe5f1e4b95946d69f2e59440928083b,rosn_sarimax_02_sarima_200,ROSN,sarima_200,"(2, 0, 0)","(0, 0, 0, 0)",c,False,0,10222.840188,...,0.922275,3.633248,4.543415,-0.026575,0.484524,3.013404,4.104011,-0.034463,0.413333,"{""run_name"": ""rosn_sarimax_02_sarima_200"", ""ti..."


## Best models


In [9]:
sarimax_runs_summary_df = (
    pd.DataFrame(all_rows)
    .sort_values(['ticker', 'val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[True, False, False, True])
    .reset_index(drop=True)
)
sarimax_runs_summary_df.to_csv(RUNS_DIR / 'sarimax_runs_summary.csv', index=False)
sarimax_runs_summary_df


,run_id,run_name,ticker,variant,order,seasonal_order,trend,use_exog,exog_count,aic,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,470cc72691e94be3baf31776a078daa3,gazp_sarimax_01_sarima_100,GAZP,sarima_100,"(1, 0, 0)","(0, 0, 0, 0)",c,False,0,12308.988348,...,0.924955,4.805111,6.054681,-0.013840,0.572619,2.361110,2.953822,-0.000257,0.535238,"{""run_name"": ""gazp_sarimax_01_sarima_100"", ""ti..."
1,48e5417d40194c26b99a6c09316476d3,gazp_sarimax_03_sarima_101,GAZP,sarima_101,"(1, 0, 1)","(0, 0, 0, 0)",c,False,0,12306.572052,...,0.924955,4.805820,6.056070,-0.014305,0.572619,2.361251,2.953722,-0.000189,0.535238,"{""run_name"": ""gazp_sarimax_03_sarima_101"", ""ti..."
2,897031592085476fb2de6eb1f3113304,gazp_sarimax_02_sarima_200,GAZP,sarima_200,"(2, 0, 0)","(0, 0, 0, 0)",c,False,0,12306.292157,...,0.925253,4.805786,6.056108,-0.014318,0.572619,2.361237,2.953731,-0.000195,0.535238,"{""run_name"": ""gazp_sarimax_02_sarima_200"", ""ti..."
3,8aa34f1038de4092b247e09a5058488c,gazp_sarimax_04_sarimax_100_exog12,GAZP,sarimax_100_exog12,"(1, 0, 0)","(0, 0, 0, 0)",c,True,12,12110.662252,...,0.921680,13.583986,14.855894,-5.103578,0.482143,18.121139,18.444397,-38.000654,0.464762,"{""run_name"": ""gazp_sarimax_04_sarimax_100_exog..."
4,c214fc224bb04602b3b3532c7a54d70c,gazp_sarimax_05_sarimax_101_exog12,GAZP,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,12093.661673,...,0.920488,15.732239,17.054354,-7.043730,0.470238,20.138843,20.547022,-47.399487,0.464762,"{""run_name"": ""gazp_sarimax_05_sarimax_101_exog..."
5,4dc5a5378a33416a8f8be37712259ea4,lkoh_sarimax_04_sarimax_100_exog12,LKOH,sarimax_100_exog12,"(1, 0, 0)","(0, 0, 0, 0)",c,True,12,9330.013670,...,0.921382,6.306895,7.746799,-3.723708,0.505952,5.981339,7.135227,-2.261620,0.385714,"{""run_name"": ""lkoh_sarimax_04_sarimax_100_exog..."
6,40d2a3305f2a473b8fcb9b3ded33c820,lkoh_sarimax_01_sarima_100,LKOH,sarima_100,"(1, 0, 0)","(0, 0, 0, 0)",c,False,0,9722.311271,...,0.919893,2.870226,3.674861,-0.062968,0.498810,3.122819,4.131089,-0.093319,0.370476,"{""run_name"": ""lkoh_sarimax_01_sarima_100"", ""ti..."
7,5345bf7c6a0548b0948364915a99d8b6,lkoh_sarimax_03_sarima_101,LKOH,sarima_101,"(1, 0, 1)","(0, 0, 0, 0)",c,False,0,9719.159409,...,0.919595,2.874929,3.679540,-0.065676,0.496429,3.125418,4.133335,-0.094508,0.370476,"{""run_name"": ""lkoh_sarimax_03_sarima_101"", ""ti..."
8,3c352d3f3e2541c9a01443ddc63a11a2,lkoh_sarimax_02_sarima_200,LKOH,sarima_200,"(2, 0, 0)","(0, 0, 0, 0)",c,False,0,9719.121497,...,0.919595,2.875123,3.679737,-0.065791,0.495238,3.125545,4.133445,-0.094566,0.370476,"{""run_name"": ""lkoh_sarimax_02_sarima_200"", ""ti..."
9,98137fc4ff75466b86e8766544a733a2,lkoh_sarimax_05_sarimax_101_exog12,LKOH,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,9396.597641,...,0.921084,13.298588,14.433159,-15.396878,0.478571,17.246240,18.440181,-20.784524,0.370476,"{""run_name"": ""lkoh_sarimax_05_sarimax_101_exog..."


In [10]:
best_models_rows = []

for ticker, summary_df in ticker_summaries.items():
    best_row = summary_df.sort_values(
        ['val_direction_accuracy', 'val_r2', 'val_rmse'],
        ascending=[False, False, True],
    ).iloc[0]
    best_models_rows.append({
        'ticker': ticker,
        'run_name': best_row['run_name'],
        'run_id': best_row['run_id'],
        'variant': best_row['variant'],
        'order': best_row['order'],
        'seasonal_order': best_row['seasonal_order'],
        'trend': best_row['trend'],
        'use_exog': best_row['use_exog'],
        'exog_count': best_row['exog_count'],
        'aic': best_row['aic'],
        'bic': best_row['bic'],
        'val_mae': best_row['val_mae'],
        'val_rmse': best_row['val_rmse'],
        'val_r2': best_row['val_r2'],
        'val_direction_accuracy': best_row['val_direction_accuracy'],
        'test_mae': best_row['test_mae'],
        'test_rmse': best_row['test_rmse'],
        'test_r2': best_row['test_r2'],
        'test_direction_accuracy': best_row['test_direction_accuracy'],
    })

best_models_summary_df = pd.DataFrame(best_models_rows).sort_values('ticker').reset_index(drop=True)
best_models_summary_df.to_csv(RUNS_DIR / 'sarimax_best_models_summary.csv', index=False)
best_models_summary_df


,ticker,run_name,run_id,variant,order,seasonal_order,trend,use_exog,exog_count,aic,bic,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy
0,GAZP,gazp_sarimax_01_sarima_100,470cc72691e94be3baf31776a078daa3,sarima_100,"(1, 0, 0)","(0, 0, 0, 0)",c,False,0,12308.988348,12327.344757,4.805111,6.054681,-0.013840,0.572619,2.361110,2.953822,-0.000257,0.535238
1,LKOH,lkoh_sarimax_04_sarimax_100_exog12,4dc5a5378a33416a8f8be37712259ea4,sarimax_100_exog12,"(1, 0, 0)","(0, 0, 0, 0)",c,True,12,9330.013670,9421.795715,6.306895,7.746799,-3.723708,0.505952,5.981339,7.135227,-2.261620,0.385714
2,ROSN,rosn_sarimax_05_sarimax_101_exog12,afa65977aa5d497b99c9eee59f073c11,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,10105.105827,10203.001908,4.248948,5.876684,-0.717475,0.617857,5.130426,6.616807,-1.689025,0.451429
3,SBER,sber_sarimax_05_sarimax_101_exog12,d986438e68fb4d1cb5d0495c3c8386db,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,9610.466041,9708.362122,3.504630,4.871988,-2.049772,0.627381,2.588095,3.733275,-3.005487,0.560952
4,TCSG,tcsg_sarimax_05_sarimax_101_exog12,1057fca8527d4e0a842b2b12782b1c15,sarimax_101_exog12,"(1, 0, 1)","(0, 0, 0, 0)",c,True,12,6606.059154,6694.190974,6.979743,7.994928,-6.348813,0.566740,26.884546,28.845126,-34.575571,0.404553
